In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
import ehtim as eh

from torchvision.transforms import v2
import data.dataset_img as ds
import torchvision
import models.model_DIReCT as mlmodel
import os
from tqdm.auto import tqdm
import mring as mr
from models.CLloss import SupConLoss
import data.imgTransforms as imgTransforms

import glob

from xlogger import *

import os

# use gpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tforms = imgTransforms.imgTransforms()

model_name = 'DIReCT_v2'
append = True
train_logger = xlogger('models/history/' + model_name + '_train.dat', append=append)
val_logger = xlogger('models/history/' + model_name + '_val.dat', append=append)
test_logger = xlogger('models/history/' + model_name + '_test.dat', append=append)


In [ ]:
# plot all images in the batch
def plot_images(images, save=False, name='images', cmap='viridis', return_axes=False, show=True):
    fig, axes = plt.subplots(2, len(images)//2, figsize=(len(images)//2*0.99, 2))
    fig.subplots_adjust(hspace=0., wspace=0.)
    axes = axes.flatten()
    for ax, img in zip(axes, images):
        ax.imshow(img.permute(1, 2, 0), cmap=cmap)
        ax.axis('off')
    if save:
        direc = 'models/history/'+name+'/'
        os.makedirs(direc, exist_ok=True)
        # check next number
        num = 0
        while glob.glob(f'{direc}{name}_{num}.png'):
            num += 1
        plt.savefig(f'{direc}{name}_{num}.png')
    if show:
        plt.show()
    if return_axes:
        return axes

def log_avgLoss(data, logger):
    # log the avg loss given array of all epochs
    data = np.sum(np.transpose(data), axis=1)/len(data)
    kwargs = {'features_loss': data[0],
              'class_loss': data[1],
              'encoder_corr': data[2],
              'pred_corr': data[3],
              'ci_loss': data[4],
              'weighted_loss': data[5]}
    logger.write(kwargs)
    return data
    


def nxcorr(outputs, labels):
    dim = int(outputs.shape[-1])
    outputs = outputs.reshape(-1, dim**2)
    labels = labels.reshape(-1, dim**2)
    
    outputs_norm = (outputs.reshape(-1, dim, dim) - torch.nanmean(outputs, axis=1).reshape(-1, 1, 1)) / torch.std(outputs, axis=1).reshape(-1, 1, 1)
    labels_norm = (labels.reshape(-1, dim, dim) - torch.nanmean(labels, axis=1).reshape(-1, 1, 1)) / torch.std(labels, axis=1).reshape(-1, 1, 1)

    fft_outputs = torch.fft.fftn(outputs_norm, s=[outputs_norm.size(d)*1 for d in [1,2]], dim=[1,2])
    fft_labels = torch.fft.fftn(labels_norm, s=[outputs_norm.size(d)*1 for d in [1,2]], dim=[1,2])

    xcorr = torch.fft.ifftn(fft_outputs * torch.conj(fft_labels), dim=[1,2])

    nxcorr_flat = xcorr.reshape(-1, dim**2)
    idx = torch.argmax(torch.abs(nxcorr_flat), dim=1)

    return idx, torch.abs(nxcorr_flat[torch.arange(nxcorr_flat.shape[0]), idx])/dim**2

def shift_image(im1, im2): # shift single im2 by idx
    idx, _ = nxcorr(im1, im2)
    im2 = torch.roll(im2, shifts=int(idx))
    return im1, im2

def shift_all(truth, imgs):
    shifted_imgs = []
    for img in imgs:
        _, shifted_img = shift_image(truth, img)
        shifted_imgs.append(shifted_img)
    return np.array(shifted_imgs)

def nxcorr_loss(outputs, labels): # assume square images
    _, xcorr = nxcorr(outputs, labels)
    loss_mean = torch.nanmean(1-xcorr)
    return loss_mean

def nxcorr_network_loss(out_img, label_img, mse_weight=1.0):
    # normalise outputs and labels
    # outputs = (outputs - torch.mean(outputs, axis=1).reshape(-1, 1)) / torch.std(outputs, axis=1).reshape(-1, 1)
    # labels = (labels - torch.mean(labels, axis=1).reshape(-1, 1)) / torch.std(labels, axis=1).reshape(-1, 1)
    mseloss = nn.MSELoss()
    mseloss_val = mseloss(out_img, label_img/(torch.max(label_img.reshape(-1, label_img.shape[-1]*label_img.shape[-2]), dim=1))[0].reshape(-1, 1, 1))
    nxcorr_val = nxcorr_loss(out_img, label_img)
    return mse_weight*mseloss_val + nxcorr_val*(1-mse_weight)


mseloss = nn.MSELoss()
ce_loss = nn.CrossEntropyLoss()
loss_weights = torch.tensor([1, 0.03, 2.0, 3.0, 0])
# loss_weights = torch.tensor([0.01, 0.03, 2.0, 3.0, 5.0])
loss_weights = torch.tensor([1, 0, 2, 3, 0])

conLoss = SupConLoss()
mse_weight = 1.0
useConLoss = False

def total_loss(outputs, input_imgs, classes, mse_weight=1.0, ci=None, clObj=None, ConLoss=False, weights=torch.tensor([1., 1., 1., 1., 0.]), print_loss=False, logger=None, epoch=1):
    features_vae, features_q, features_ci, recon_img, pred_img, pred_class = outputs
    # features_vae = features_vae.view(features_vae.shape[0], -1)
    # features_ci = features_ci.view(features_ci.shape[0], -1)
    
    features_loss = mseloss(features_ci, features_q)
    if ConLoss:
        features_ci = features_ci.reshape(features_ci.shape[0], -1)
        features_q = features_q.reshape(features_q.shape[0], -1)
        features_ci = (features_ci/torch.norm(features_ci, dim=1).reshape(-1, 1)).reshape(features_ci.shape[0], 1, -1)
        features_q = (features_q/torch.norm(features_q, dim=1).reshape(-1, 1)).reshape(features_q.shape[0], 1, -1)
        features = torch.cat((features_ci, features_q), dim=1)
        features_loss = conLoss(features, torch.arange(features.shape[0]).to(device))

    # class_loss = ce_loss(pred_class, classes)
    class_loss = torch.tensor(0).to(device)

    # first channel only
    recon_img = recon_img[:,0]
    pred_img = pred_img[:,0]
    input_imgs = input_imgs[:,0]
    encoder_corr = nxcorr_network_loss(recon_img, input_imgs, mse_weight=mse_weight)
    pred_corr = nxcorr_network_loss(pred_img, input_imgs, mse_weight=mse_weight)

    ci_loss = torch.tensor(0.0).to(device)
    if clObj != None and ci != None and weights[-1] != 0:
        out_ci = clObj.FTCI(pred_img, return_combined=False).to(device)
        ci_loss = mseloss(out_ci, ci[:, :out_ci.shape[-1]])
        pass


    total = weights[0]*features_loss + weights[1]*class_loss + weights[2]*encoder_corr + weights[3]*pred_corr + weights[4]*ci_loss

    losses_individual = [features_loss.cpu().detach().numpy(), 
                         class_loss.cpu().detach().numpy(), 
                         encoder_corr.cpu().detach().numpy(), 
                         pred_corr.cpu().detach().numpy(), 
                         ci_loss.cpu().detach().numpy()]
    losses_weighted = np.array([weights[i]*losses_individual[i] for i in range(len(weights))] + [total.cpu().detach()])
    
    if print_loss:
        print(f'features_loss: {features_loss}, class_loss: {class_loss}, encoder_corr: {encoder_corr}, pred_corr: {pred_corr}, ci_loss: {ci_loss}')
    if logger:
        kwargs = {'features_loss': features_loss.cpu().detach().numpy(), 
                  'class_loss': class_loss.cpu().detach().numpy(), 
                  'encoder_corr': encoder_corr.cpu().detach().numpy(), 
                  'pred_corr': pred_corr.cpu().detach().numpy(),
                  'ci_loss': ci_loss.cpu().detach().numpy(),
                  'weighted_loss': total.cpu().detach().numpy()}
        logger.write(kwargs)
        

    return total, losses_individual, losses_weighted

In [ ]:
# Load initial data

combine_ci_vis = False

ehtim=True
tint_sec = 5   
tadv_sec = 600
tstart_hr = 0
tstop_hr = 24
# psize = 7.757018897750619e-12 * 2
psize = 1.7044214966184275e-11
bw_hz = [230E9]#, 345E9]

# uvfits_files = ['../ngEHT_Challenges/Challenge_1/synthetic_data/M87_eht2022_230_thnoise.uvfits']
uvfits_files = None

batch_size = 32

data_dir = 'data/datasets/v3/'
mnames = ['gauss', 'disk', 'ellipse', 'ring', 'mring', 'disk2', 'gauss2', 'cifar10-128']
# mnames = ['gauss', 'disk', 'ellipse', 'ring', 'mring', 'disk2', 'gauss2']
# mnames = ['gauss', 'disk', 'ellipse']
# mnames = ['cifar10-128']
train_data = ds.ImgDataset(['data/datasets/v3/train/train_mring.npy'], transform=tforms.train_transforms,
                           tint_sec=tint_sec, tadv_sec=tadv_sec, tstart_hr=tstart_hr, tstop_hr=tstop_hr, bw_hz=bw_hz, psize=psize,
                           uvfits_files=uvfits_files
                           )
train = DataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=True)

val_data = ds.ImgDataset(['data/datasets/v3/val/val_mring.npy'], transform=tforms.val_transforms,
                            tint_sec=tint_sec, tadv_sec=tadv_sec, tstart_hr=tstart_hr, tstop_hr=tstop_hr, bw_hz=bw_hz, psize=psize,
                            uvfits_files=uvfits_files
                            )

val = DataLoader(val_data, batch_size=batch_size, shuffle=True, drop_last=True)

dataiter = iter(val)
static_val = next(dataiter)

clObj = train.dataset.closure

ci_shape = clObj.FTCI(static_val[0]).shape[-1]
data_dim = clObj.FTCI(static_val[0], return_combined=combine_ci_vis).shape[-1]

print(data_dim)
# plot_images(static_val[0])


In [ ]:
# Initialise Model

load = True

# initialise weights
def weights_init(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)

if load:
    model = torch.load('models/saved_models/'+model_name+'.pt')
    model = model.module
else: 
    model = mlmodel.DIReCT(data_dim=data_dim, k_classes=1)
    # model.apply(weights_init)

if False: # WARNING: Replaces all parameters but the transformer!
    # load encoder, decoder, classifier from pre-trained model
    pretrained = torch.load('models/saved_models/AttnAE_all_v3.pt').named_parameters()
    dict_params = dict(pretrained)
    for name, param in model.named_parameters():
        if 'ci_xtrans' in name or 'classifier' in name:
            continue
        if name in dict_params:
            param.data = dict_params[name].data
model.to(device);
# model = nn.DataParallel(model, device_ids=[2,3])


In [ ]:
# Loads large datasets!!

transforms_list = [tforms.train_transforms] * int(len(mnames)-1) + [tforms.cifar_train_transforms]
val_transforms_list = [tforms.val_transforms] * int(len(mnames)-1) + [tforms.cifar_val_transforms]

# transforms_list = [tforms.train_transforms] * int(len(mnames))
# val_transforms_list = [tforms.val_transforms] * int(len(mnames))


filenames = np.array([data_dir + 'train/train_' + m + '.npy' for m in mnames])
train_data = ds.ImgDataset(filenames, transform=tforms.cifar_train_transforms, transform_list=transforms_list,
                           tint_sec=tint_sec, tadv_sec=tadv_sec, tstart_hr=tstart_hr, tstop_hr=tstop_hr, bw_hz=bw_hz, psize=psize,
                           uvfits_files=uvfits_files
                           )

filenames = np.array([data_dir + 'val/val_' + m + '.npy' for m in mnames])
val_data = ds.ImgDataset(filenames, transform=tforms.cifar_val_transforms, transform_list=val_transforms_list,
                         tint_sec=tint_sec, tadv_sec=tadv_sec, tstart_hr=tstart_hr, tstop_hr=tstop_hr, bw_hz=bw_hz, psize=psize,
                         uvfits_files=uvfits_files
                         )


print('Img Shape:', train_data[0][0].shape)
print('CI Shape:', train_data[0][1].shape)
print('Class Shape:', train_data[0][2].shape)

train = DataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=True)#, num_workers=16)
val = DataLoader(val_data, batch_size=batch_size, shuffle=True, drop_last=True)#, num_workers=16)

dataiter = iter(train)
data = next(dataiter)

print()

print('Img Data Shape:', data[0].shape)
# print('CI Data Shape:', data[1].shape)
print('CI Data Shape:', clObj.FTCI(data[0]).reshape(batch_size, -1).shape)
print('Class Data Shape:', data[2].shape)  


# single validation batch
dataiter = iter(val)
static_val = next(dataiter)

plot_images(static_val[0])





In [ ]:
# Check out the loss

criterion = total_loss
i = 0 
for i, data in enumerate(train, 0):
    imgs, ci, cls = data
    ci = torch.tensor(clObj.FTCI(imgs, return_combined=combine_ci_vis).reshape(batch_size, -1), dtype=torch.float32)
    plot_images(imgs.detach().cpu())
    imgs, ci, cls = imgs.to(device), ci.to(device), cls.to(device)
    outputs = model(imgs, ci)
    plot_images(outputs[3].detach().cpu())
    plot_images(outputs[4].detach().cpu())
    loss, _, _ = criterion(outputs, imgs, cls, mse_weight=mse_weight, ci=ci, clObj=clObj, ConLoss=useConLoss, print_loss=True, weights=loss_weights, logger=None)
    i += 1


    print('loss:', loss)

    test = imgs.detach().cpu().numpy()[0]

    break


In [ ]:
if False:

    nepochs = 50
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    criterion = total_loss
    l1 = lambda epoch: 0.99 ** epoch
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=l1)

    min_val_loss = np.inf

    for epoch in range(nepochs):
        model.train()
        train_loss = []
        for data in tqdm(train):
            imgs, ci, cls  = data
            ci = torch.tensor(clObj.FTCI(imgs, return_combined=combine_ci_vis).reshape(imgs.shape[0], -1), dtype=torch.float32)
            
            imgs, ci, cls = imgs.to(device), ci.to(device), cls.to(device)
            optimizer.zero_grad()
            outputs = model(imgs, ci)
            loss, loss_individual, loss_weighted = criterion(outputs, imgs, cls, mse_weight=mse_weight, ci=ci, clObj=clObj, ConLoss=useConLoss, 
                                                            weights=loss_weights, logger=test_logger, epoch=epoch)
            loss.backward()
            optimizer.step()
            train_loss.append(loss_weighted)
        train_loss = log_avgLoss(train_loss, train_logger)
        model.eval()
        val_loss = []
        for data in val:
            imgs, ci, cls  = data       
            ci = torch.tensor(clObj.FTCI(imgs, return_combined=combine_ci_vis).reshape(imgs.shape[0], -1), dtype=torch.float32)

            imgs, ci, cls = imgs.to(device), ci.to(device), cls.to(device)
            outputs = model(imgs, ci)
            loss, loss_individual, loss_weighted = criterion(outputs, imgs, cls, mse_weight=mse_weight, ci=ci, clObj=clObj, ConLoss=useConLoss, 
                                                            weights=loss_weights, logger=None, epoch=epoch)
            val_loss.append(loss_weighted)
        val_loss = log_avgLoss(val_loss, val_logger)

        if val_loss[-1] < min_val_loss: # save model
            min_val_loss = val_loss[-1]
            torch.save(model, 'models/saved_models/'+model_name+'.pt')
            pass
        
        # save checkpoint
        if epoch % 10 == 0:
            torch.save(model, 'models/saved_models/'+model_name+'_'+str(epoch)+'.pt')
            pass

        # plot images in the last batch every n epochs
        if (epoch+1) % 5 == 0 or epoch == 0:
            pass
            # plots result
            imgs, ci, cls = static_val
            ci = torch.tensor(train_data.closure.FTCI(imgs, return_combined=combine_ci_vis).reshape(batch_size, -1), dtype=torch.float32)
            
            features_vae, features_q, features_ci, recon_img, pred_img, pred_class = model(imgs.to(device), ci.to(device))
            plot_images(imgs.cpu().detach())
            plot_images(recon_img.cpu().detach()) 
            plot_images(pred_img.cpu().detach(), save=False, name=model_name) 
            
        print(f'Epoch {epoch+1}, Train loss: {train_loss[-1]}, Val loss: {val_loss[-1]}, lr: {scheduler.get_last_lr()[0]}')
        scheduler.step()
    

In [ ]:
# plot loss history

train_loss = read_xlogfile('models/history/'+model_name+'_train.dat')
val_loss = read_xlogfile('models/history/'+model_name+'_val.dat')

l_weights = loss_weights.numpy()

fig, ax = plt.subplots(1,2, figsize=(10,5))
plot_from = 0
ax[0].plot(train_loss['weighted_loss'][plot_from:], label='Weighted')
ax[0].plot(l_weights[0]*train_loss['features_loss'][plot_from:], label='Features')
ax[0].plot(l_weights[1]*train_loss['class_loss'][plot_from:], label='Class')
ax[0].plot(l_weights[2]*train_loss['encoder_corr'][plot_from:], label='Encoder')
ax[0].plot(l_weights[3]*train_loss['pred_corr'][plot_from:], label='Prediction')

plot_from = 0
ax[1].plot(val_loss['weighted_loss'][plot_from:], label='Weighted')
ax[1].plot(l_weights[0]*val_loss['features_loss'][plot_from:], label='Features')
ax[1].plot(l_weights[1]*val_loss['class_loss'][plot_from:], label='Class')
ax[1].plot(l_weights[2]*val_loss['encoder_corr'][plot_from:], label='Encoder')
ax[1].plot(l_weights[3]*val_loss['pred_corr'][plot_from:], label='Prediction')

# epoch of min val loss
min_val_loss = np.min(val_loss['weighted_loss'][0:])
min_val_loss_idx = np.argmin(val_loss['weighted_loss'][0:])
print(f'Min val loss: {min_val_loss}, epoch: {min_val_loss_idx+0}')

ax[0].legend()
ax[1].legend()

